# Streamflow Data Cleaning: USGS Iowa Daily Discharge

Cleans the **daily mean discharge** time series for the USGS Iowa streamgages
into a tidy, type-safe table keyed on (`site_no`, `date`), ready to join to the
gage metadata and to water-quality observations.

**Input:**  `data/tabular/01_raw/streamflow/usgs-iowa-discharge.csv`
**Output:** `data/tabular/02_clean/streamflow/usgs-iowa-discharge-clean.csv`

Each row is one gage-day of mean streamflow:

| raw column      | meaning                                              | units |
|-----------------|------------------------------------------------------|-------|
| `site_no`       | USGS site number (joins to the gauges table)         | id    |
| `date`          | observation date (midnight-UTC daily value)          | date  |
| `discharge_cfs` | daily mean discharge (USGS parameter 00060)          | ft³/s |
| `discharge_cd`  | USGS data-value qualification code                   | code  |

The qualification code is a comma-separated set: an approval status — `A`
(approved) or `P` (provisional) — optionally followed by qualifiers `e`
(estimated), `R` (revised), `Ice` (ice-affected) or `<` (value is an upper bound).

**Cleaning steps:**

1. Load the raw extract, `site_no` as a string.
2. **Parse `date`** to a plain calendar date (the time/zone is a constant
   midnight-UTC stamp on a daily series).
3. **Replace the `-999999` sentinel** discharge with null (USGS no-data marker,
   here all ice-affected provisional days).
4. **Clamp** the few tiny negative estimated values to `0` — discharge is
   non-negative.
5. **Drop non-observation rows** that carry no discharge value (pure gaps).
6. **Validate** the qualification-code vocabulary and value ranges.
7. **De-duplicate** on (`site_no`, `date`).
8. **Sort & write** the tidy table to `02_clean`.

> The cleaned `discharge_cd` is retained as a provenance/quality flag rather than
> filtered on, so downstream modeling can decide whether to exclude provisional
> (`P`) or estimated (`e`) days.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "streamflow"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "streamflow"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)

Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction
Raw dir:   /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/01_raw/streamflow
Clean dir: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/streamflow


## Step 1 — Load

~560K gage-days across the daily-discharge sites. `site_no` is read as a string
so it joins cleanly to the gauges table; `date` is parsed in the next step.

In [2]:
df = pd.read_csv(
    RAW_DIR / "usgs-iowa-discharge.csv",
    dtype={"site_no": "string", "discharge_cd": "string"},
)
n_raw = len(df)
print(f"Loaded {n_raw:,} rows across {df['site_no'].nunique()} sites")
print("Columns:", list(df.columns))
print("\nNulls per column:")
print(df.isna().sum().to_string())
df.head()

Loaded 560,896 rows across 155 sites
Columns: ['site_no', 'date', 'discharge_cfs', 'discharge_cd']

Nulls per column:
site_no             0
date                0
discharge_cfs    4020
discharge_cd     4020


,site_no,date,discharge_cfs,discharge_cd
0,05387440,2015-01-01 00:00:00+00:00,127.0,"A, e"
1,05387440,2015-01-02 00:00:00+00:00,169.0,"A, e"
2,05387440,2015-01-03 00:00:00+00:00,173.0,"A, e"
3,05387440,2015-01-04 00:00:00+00:00,154.0,"A, e"
4,05387440,2015-01-05 00:00:00+00:00,124.0,"A, e"


## Step 2 — Parse the date

Every timestamp is a constant `00:00:00+00:00` — these are daily values, not
sub-daily, so the time and zone carry no information. Parse to a plain calendar
`date`.

In [3]:
ts = pd.to_datetime(df["date"], utc=True)
assert (ts.dt.time.astype(str) == "00:00:00").all(), "non-midnight timestamp"
df["date"] = ts.dt.date
print("Date range:", df["date"].min(), "→", df["date"].max())

Date range: 2015-01-01 → 2025-12-31


## Step 3 — Replace the `-999999` no-data sentinel

USGS publishes `-999999` as a no-data marker; here all such rows are
provisional, ice-affected days (`P, Ice`). Convert the sentinel to a true null so
it can't poison statistics.

In [4]:
SENTINEL = -999999
n_sentinel = (df["discharge_cfs"] == SENTINEL).sum()
print(f"Sentinel ({SENTINEL}) rows: {n_sentinel}")
print(df.loc[df["discharge_cfs"] == SENTINEL, "discharge_cd"].value_counts().to_string())
df.loc[df["discharge_cfs"] == SENTINEL, "discharge_cfs"] = np.nan

Sentinel (-999999) rows: 26
discharge_cd
P, Ice    26


## Step 4 — Clamp tiny negative estimates

After removing the sentinel, a few approved, estimated values are slightly
negative (~`-0.1` to `-0.8` ft³/s) — backwater/estimation artifacts near zero
flow. Discharge is physically non-negative, so clamp them to `0`. Adding `0.0`
also folds any IEEE negative zeros (`-0.0`) carried in from the raw export into a
plain `0.0`.

In [5]:
neg = df["discharge_cfs"] < 0
print(f"Negative values to clamp: {neg.sum()}")
print(df.loc[neg, ["site_no", "date", "discharge_cfs", "discharge_cd"]].to_string())
df.loc[neg, "discharge_cfs"] = 0.0
df["discharge_cfs"] = df["discharge_cfs"] + 0.0   # normalize -0.0 → 0.0

Negative values to clamp: 4
         site_no        date  discharge_cfs discharge_cd
293840  05473450  2017-12-26          -0.14         A, e
293847  05473450  2018-01-02          -0.46         A, e
293848  05473450  2018-01-03          -0.78         A, e
293849  05473450  2018-01-04          -0.68         A, e


## Step 5 — Drop non-observation rows

A row whose `discharge_cfs` is null (the original gaps plus the converted
sentinels) records no measurement, so it is dropped from the measurement table.
Genuine gaps are recoverable from the gage's date range if ever needed.

In [6]:
n_null = df["discharge_cfs"].isna().sum()
print(f"Non-observation rows to drop: {n_null:,}")
df = df.dropna(subset=["discharge_cfs"]).reset_index(drop=True)
print(f"Rows remaining: {len(df):,}")

Non-observation rows to drop: 4,046
Rows remaining: 556,850


## Step 6 — Validate codes and ranges

Confirm every `discharge_cd` is drawn from the expected USGS vocabulary
(approval status + optional qualifiers) and that discharge is now non-negative
and finite.

In [7]:
VALID_TOKENS = {"A", "P", "e", "R", "Ice", "<"}
tokens = (
    df["discharge_cd"].dropna().str.split(",").explode().str.strip().unique()
)
bad = set(tokens) - VALID_TOKENS
assert not bad, f"unexpected discharge_cd tokens: {bad}"

assert (df["discharge_cfs"] >= 0).all(), "negative discharge remains"
assert np.isfinite(df["discharge_cfs"]).all(), "non-finite discharge"

print("discharge_cd value counts:")
print(df["discharge_cd"].value_counts(dropna=False).to_string())
print("\ndischarge_cfs (ft³/s):")
print(df["discharge_cfs"].describe().to_string())
print("\nAll code/range checks passed.")

discharge_cd value counts:
discharge_cd
A       461147
A, e     94022
A, R      1591
P           64
P, e        25
A, <         1

discharge_cfs (ft³/s):
count    556850.000000
mean       3240.479769
std       12733.194317
min           0.000000
25%          69.200000
50%         346.000000
75%        1580.000000
max      534000.000000

All code/range checks passed.


## Step 7 — De-duplicate

(`site_no`, `date`) should uniquely identify a daily observation. Drop exact
duplicate rows and confirm the composite key is unique.

In [8]:
df = df.drop_duplicates()
dup_keys = df.duplicated(subset=["site_no", "date"]).sum()
print(f"Duplicate (site_no, date) pairs: {dup_keys}")
assert dup_keys == 0, "duplicate (site_no, date) remains"

Duplicate (site_no, date) pairs: 0


## Step 8 — Sort & write

Order columns key-first, sort by (`site_no`, `date`), and write the tidy table to
`02_clean`.

In [9]:
ordered = ["site_no", "date", "discharge_cfs", "discharge_cd"]
df = df[ordered].sort_values(["site_no", "date"]).reset_index(drop=True)

out_path = CLEAN_DIR / "usgs-iowa-discharge-clean.csv"
df.to_csv(out_path, index=False)
print(f"Wrote {len(df):,} rows × {df.shape[1]} cols (from {n_raw:,}) to:")
print(" ", out_path.relative_to(REPO_ROOT))
df.head()

Wrote 556,850 rows × 4 cols (from 560,896) to:
  data/tabular/02_clean/streamflow/usgs-iowa-discharge-clean.csv


,site_no,date,discharge_cfs,discharge_cd
0,05387440,2015-01-01,127.0,"A, e"
1,05387440,2015-01-02,169.0,"A, e"
2,05387440,2015-01-03,173.0,"A, e"
3,05387440,2015-01-04,154.0,"A, e"
4,05387440,2015-01-05,124.0,"A, e"
